## Population.ipynb
Author Cathal Redmond
# Date 2025-Oct-23


In [141]:
import pandas as pd

In [142]:
url = "https://ws.cso.ie/public/api.restful/PxStat.Data.Cube_API.ReadDataset/FY006A/CSV/1.0/en" 
df = pd.read_csv(url)
df.tail(3)

,STATISTIC,Statistic Label,TLIST(A1),CensusYear,C02199V02655,Sex,C02076V03371,Single Year of Age,C03789V04537,Administrative Counties,UNIT,VALUE
9789,FY006AC01,Population,2022,2022,2,Female,650,100 years and over,2ae19629-149d-13a3-e055-000000000001,Cavan County Council,Number,12
9790,FY006AC01,Population,2022,2022,2,Female,650,100 years and over,2ae19629-14a4-13a3-e055-000000000001,Donegal County Council,Number,31
9791,FY006AC01,Population,2022,2022,2,Female,650,100 years and over,2ae19629-1495-13a3-e055-000000000001,Monaghan County Council,Number,7


Need to strip the gender from this dataframe 

In [143]:
#df = df[df["Sex"] != "Female"]
#df = df[df["Sex"] != "Male"]
df = df[df["Sex"] != "Both sexes"]
df.tail(3)


,STATISTIC,Statistic Label,TLIST(A1),CensusYear,C02199V02655,Sex,C02076V03371,Single Year of Age,C03789V04537,Administrative Counties,UNIT,VALUE
9789,FY006AC01,Population,2022,2022,2,Female,650,100 years and over,2ae19629-149d-13a3-e055-000000000001,Cavan County Council,Number,12
9790,FY006AC01,Population,2022,2022,2,Female,650,100 years and over,2ae19629-14a4-13a3-e055-000000000001,Donegal County Council,Number,31
9791,FY006AC01,Population,2022,2022,2,Female,650,100 years and over,2ae19629-1495-13a3-e055-000000000001,Monaghan County Council,Number,7


Now we can do code that will prep this data for analysis. We need to take the column names and assign them into a list


In [144]:
headers = df.columns.tolist()
headers

['STATISTIC',
 'Statistic Label',
 'TLIST(A1)',
 'CensusYear',
 'C02199V02655',
 'Sex',
 'C02076V03371',
 'Single Year of Age',
 'C03789V04537',
 'Administrative Counties',
 'UNIT',
 'VALUE']

get rid of the columns we will not be using 


In [145]:
drop_col_list = ['STATISTIC', 'Statistic Label','TLIST(A1)','CensusYear','C02199V02655','Administrative Counties','C02076V03371','C03789V04537','UNIT']
df.drop(columns=drop_col_list, inplace=True)
df = df[df["Single Year of Age"] != "All ages"]
df['Single Year of Age'] = df['Single Year of Age'].str.replace('Under 1 year', '0')
df['Single Year of Age'] = df['Single Year of Age'].str.replace('\D', '', regex=True)

df['Single Year of Age']=df['Single Year of Age'].astype('int64')
df['VALUE']=df['VALUE'].astype('int64')
print (df)
df.info()
#export full file to local machine
df.to_csv("All_Rows_By_Single_year_of_birth.csv")

         Sex  Single Year of Age  VALUE
3296    Male                   0  29610
3297    Male                   0    346
3298    Male                   0   3188
3299    Male                   0   1269
3300    Male                   0   2059
...      ...                 ...    ...
9787  Female                 100      7
9788  Female                 100      9
9789  Female                 100     12
9790  Female                 100     31
9791  Female                 100      7

[6464 rows x 3 columns]
<class 'pandas.core.frame.DataFrame'>
Index: 6464 entries, 3296 to 9791
Data columns (total 3 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   Sex                 6464 non-null   object
 1   Single Year of Age  6464 non-null   int64 
 2   VALUE               6464 non-null   int64 
dtypes: int64(2), object(1)
memory usage: 202.0+ KB


<>:5: SyntaxWarning: invalid escape sequence '\D'
<>:5: SyntaxWarning: invalid escape sequence '\D'
C:\Users\catha\AppData\Local\Temp\ipykernel_4080\2397195758.py:5: SyntaxWarning: invalid escape sequence '\D'
  df['Single Year of Age'] = df['Single Year of Age'].str.replace('\D', '', regex=True)


In [146]:
df_anal = pd.pivot_table(df,'VALUE', "Single Year of Age", "Sex")
print (df_anal.head(3))

#export full file to local machine
df_anal.to_csv("population_for_analysis.csv")

Sex                    Female       Male
Single Year of Age                      
0                   1761.6250  1850.6250
1                   1721.5625  1804.6875
2                   1810.8750  1889.7500


# Time for Descriptive Stats

In [147]:
headers = list(df_anal.columns)
sex = headers[0]
sex 

'Female'

OK now we need to get the Weighted Mean. 
Formula is Weighted mean = Sum(age*population at age)/sum(populations at age)

In [148]:
number_people = df_anal[sex].sum()
number_people

162786.875

In [149]:
df_anal

Sex,Female,Male
Single Year of Age,,
0,1761.6250,1850.6250
1,1721.5625,1804.6875
2,1810.8750,1889.7500
3,1842.6875,1937.5625
4,1863.6875,1980.3750
...,...,...
96,59.7500,20.4375
97,45.7500,13.5625
98,30.7500,8.1250


In [150]:
sex_gap_at_age = df_anal['Female']-df_anal['Male']
sex_gap_at_age

Single Year of Age
0      -89.0000
1      -83.1250
2      -78.8750
3      -94.8750
4     -116.6875
         ...   
96      39.3125
97      32.1875
98      22.6250
99      14.4375
100     26.8750
Length: 101, dtype: float64

calculate what the cumulative ages are 

In [151]:
cumages = df_anal[sex].mul(df_anal.index, axis = 0).sum()
cumages

6338887.6875

In [152]:
weighted_mean = cumages/number_people
weighted_mean

38.9397958987787

Weighted Median

In [153]:
cumsum = df_anal[sex].cumsum()
cumsum

Single Year of Age
0        1761.6250
1        3483.1875
2        5294.0625
3        7136.7500
4        9000.4375
          ...     
96     162652.8750
97     162698.6250
98     162729.3750
99     162750.3750
100    162786.8750
Name: Female, Length: 101, dtype: float64

In [154]:
cutoff = df_anal[sex].sum()/2
cutoff

81393.4375

In [155]:
df_anal[sex][cumsum>=cutoff].index[0]

39

Weighted Standard Deviation

In [156]:
import numpy as np 
w_mean = np.average(df_anal.index, weights = df_anal[sex])
w_mean

38.9397958987787

In [157]:
w_variance = np.average((df_anal.index - w_mean)**2, weights = df_anal[sex])
w_variance

528.953520736661

In [158]:
w_std = np.sqrt(w_variance)
w_std

22.998989559036303